# EX: Routing Optimization Evaluation

In this exercise, we will simulate passing node data to two different search agents operating in a tactical environment. We will observe how a Greedy routing agent falls into a geographical trap, while an A* routing agent successfully calculates the optimal path by balancing both exact cost and heuristics.

**Steps Performed:**

* **Define a simulation dictionary** that acts as our state space graph, detailing waypoints, the true fuel costs between them, and their estimated heuristic distances to the target.

* **Execute Greedy Search** and observe how it gets baited into a high-cost path (Test 1).

* **Execute A* Search** and observe how factoring in exact cost ($g(n)$) forces the agent to reconsider and find the optimal route (Test 2).

```{mermaid}
graph LR
    S((Start<br>h = 10))
    A((Waypoint A<br>h = 2))
    B((Waypoint B<br>h = 5))
    G((Goal<br>h = 0))

    S -- "Cost: 1" --> A
    S -- "Cost: 2" --> B
    A -- "Cost: 10" --> G
    B -- "Cost: 2" --> G

    %% Styling for light/dark mode compatibility
    style S fill:transparent,stroke:#3498db,stroke-width:3px
    style G fill:transparent,stroke:#2ecc71,stroke-width:3px
    style A fill:transparent,stroke:#95a5a6,stroke-width:2px
    style B fill:transparent,stroke:#95a5a6,stroke-width:2px
```

In [1]:
# --- Scenario Data ---
# Graph structure: {Node: {'neighbors': {Neighbor: Fuel_Cost}, 'h': Heuristic_Distance}}
tactical_map = {
    'Start': {'neighbors': {'A': 1, 'B': 2}, 'h': 10},
    'A': {'neighbors': {'Goal': 10}, 'h': 2},
    'B': {'neighbors': {'Goal': 2}, 'h': 5},
    'Goal': {'neighbors': {}, 'h': 0}
}

def simulate_greedy(graph, start, goal):
    """Simulates Greedy Best-First Search using f(n) = h(n)"""
    current = start
    path = [current]
    total_cost = 0
    
    while current != goal:
        neighbors = graph[current]['neighbors']
        # Greedy looks ONLY at the neighbor's heuristic h(n)
        next_node = min(neighbors.keys(), key=lambda n: graph[n]['h'])
        
        total_cost += neighbors[next_node]
        current = next_node
        path.append(current)
        
    return path, total_cost

def simulate_astar(graph, start, goal):
    """Simulates A* Search using f(n) = g(n) + h(n)"""
    # Open set format: {Node: (g_cost, f_cost, path_history)}
    open_set = {start: (0, graph[start]['h'], [start])}
    
    while open_set:
        # A* looks at the lowest f(n) in the entire open set
        current = min(open_set.keys(), key=lambda n: open_set[n][1])
        g_current, f_current, path = open_set.pop(current)
        
        if current == goal:
            return path, g_current
            
        for neighbor, cost in graph[current]['neighbors'].items():
            g_new = g_current + cost
            f_new = g_new + graph[neighbor]['h']
            # Add to open set for future evaluation
            open_set[neighbor] = (g_new, f_new, path + [neighbor])
            
    return None, 0

print("--- TEST 1: Greedy Search (f = h) ---")
greedy_path, greedy_cost = simulate_greedy(tactical_map, 'Start', 'Goal')
print(f"PATH TAKEN: {' -> '.join(greedy_path)}")
print(f"TOTAL FUEL COST: {greedy_cost}\n")

print("--- TEST 2: A* Search (f = g + h) ---")
astar_path, astar_cost = simulate_astar(tactical_map, 'Start', 'Goal')
print(f"PATH TAKEN: {' -> '.join(astar_path)}")
print(f"TOTAL FUEL COST: {astar_cost}")

--- TEST 1: Greedy Search (f = h) ---
PATH TAKEN: Start -> A -> Goal
TOTAL FUEL COST: 11

--- TEST 2: A* Search (f = g + h) ---
PATH TAKEN: Start -> B -> Goal
TOTAL FUEL COST: 4


# Interpreting the Results

```{figure} ../../figures/informed-search-lab.png
---
width: 100%
align: center
name: informed-search-lab
---
Heuristic example comparing Greedy Search and Ausing $A^*$.
```

In Test 1, the Greedy agent looked forward from the 'Start' waypoint, saw that Waypoint A had a much lower heuristic estimate ($h=2$) than Waypoint B ($h=5$), and immediately committed to it. It ignored the catastrophic downstream fuel cost of 10, resulting in a suboptimal mission cost of 11.

In Test 2, the A* agent initially explored Waypoint A just like the Greedy agent. However, as it calculated the final hop to the Goal, its mathematical evaluation function $f(n)$ spiked to 11. Because it was tracking the overall state space, it reverted its decision, explored Waypoint B ($f=7$), and discovered the significantly cheaper, optimal path requiring a fuel cost of only 4.